[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/lab4.ipynb)

# Lab 4: The Coefficients That Stayed Zero
## Penalized Regression & Biological Interpretation

**Day 2, 11:00–12:00** | Learning Outcomes:
- **LO2** — apply penalized regression (ridge / LASSO / elastic net) to high-dimensional biological data.
- **LO3** — critically evaluate analysis code and AI-generated output.

This lab applies **Lecture 4** (penalized regression) — the regression analogue of the shrinkage estimation from Lecture 2. The connections:

- **Lecture 2** shrank gene-level *means* (Stein / James–Stein) and gene-level *variances* (limma's eBayes).
- **Lecture 4** shrinks gene-level *regression coefficients*: ridge shrinks all to small values; LASSO shrinks most to exactly zero (sparse selection); elastic net combines the two for grouped predictors.

In this lab you will work through a complete penalized regression analysis: data preparation, model fitting, model selection, stability analysis, and biological interpretation. Use AI to generate code where helpful, but focus on the *decisions*: which method, which parameters, what do the results mean?

::: {.callout-note}
## How this lab uses AI

This lab integrates AI at several steps, not just at the end. Two kinds of AI exercises appear:

- **Single-shot critiques** — you prompt, paste the response, and evaluate it against named criteria.
- **Multi-turn conversations** — you prompt, critique the response, write a follow-up prompt that addresses your critique, paste the refined response, and write a final assessment. *At least one multi-turn exercise per lab is required.* In this lab, the **Part 5 (bootstrap stability)** exercise is the multi-turn one.

Use any AI assistant (the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, ChatGPT, or another). Note which AI you used for each prompt.
:::

The task: **build a sparse gene signature that distinguishes ALL from AML.**

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.exceptions import ConvergenceWarning
import warnings

# Workshop convention: deterministic seed.
np.random.seed(2026)

# Show convergence warnings — they are pedagogically valuable in Part 1
# (we suppress only the FutureWarnings about API changes to keep output clean).
warnings.filterwarnings('default', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12


In [ ]:
# Load the Golub expression matrix and metadata from the workshop repo (CSV — same source as Labs 1, 2, 3).
expr_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_expression.csv"
meta_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_metadata.csv"

expr_genes_x_samples = pd.read_csv(expr_url, index_col=0)
meta = pd.read_csv(meta_url)

assert expr_genes_x_samples.shape == (3051, 72), \
    f"Unexpected expression shape: {expr_genes_x_samples.shape} (expected (3051, 72))"
assert list(expr_genes_x_samples.columns) == list(meta["sample_id"]), \
    "Sample IDs do not match between expression and metadata."

# samples x genes for sklearn
X_raw = expr_genes_x_samples.values.T          # 72 x 3051
gene_names = list(expr_genes_x_samples.index)
assert len(gene_names) == 3051

# Binary outcome: 0 = ALL, 1 = AML
labels = meta["subtype"]
y = (labels == 'AML').astype(int).values

# Standardize features (zero mean, unit variance per gene)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Loaded Golub leukemia data from the workshop CSVs.")
print(f"X: {X.shape[0]} samples x {X.shape[1]} genes")
print(f"y: {int(np.sum(y == 0))} ALL, {int(np.sum(y == 1))} AML")
print(f"Ratio p/n: {X.shape[1] / X.shape[0]:.1f}")
print("Note: y == 1 means AML; sign of fitted coefficients is relative to this encoding.")


---
## Part 1: Why OLS Fails

Before we add any penalty, let's try to fit logistic regression without regularization on all 3,051 genes. Watch it fail or overfit.

### Exercise 1.0 — AI explains why unregularized logistic regression fails (single-shot)

Before you run the next cell, ask an AI assistant to predict what will happen.

> "Reply in two short sentences only (no .docx file). I'm about to fit unregularized logistic regression on a microarray dataset with $p = 3{,}051$ gene-expression features and only $n = 72$ samples. The training accuracy will be near 100% but the 10-fold cross-validation accuracy will be much lower. Explain *why* in terms of: (1) the rank of $X^\top X$ when $p > n$, (2) what `sklearn`'s solver picks among the infinitely many separating hyperplanes, and (3) what training accuracy vs CV accuracy tells you about the difference between memorization and generalization."

**AI's response (paste here):**


**Your critique** — common failure modes:

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did the AI name that $X^\top X$ is *singular* when $p > n$ (rank at most $n$)? | | |
| Did the AI explain *why* the solver can perfectly separate the training data (infinitely many separating hyperplanes; sklearn picks the min-norm one)? | | |
| Did the AI distinguish training accuracy $\to 1$ (memorization) from CV accuracy collapse (no generalization)? | | |

**One-sentence overall assessment** — did the AI name all three failure modes correctly?

In [ ]:
np.random.seed(2026)

# Unregularized logistic regression — no penalty
lr_none = LogisticRegression(penalty=None, max_iter=10000, random_state=2026)
lr_none.fit(X, y)

# Training accuracy
train_acc = lr_none.score(X, y)

# Cross-validation accuracy
cv_scores = cross_val_score(lr_none, X, y, cv=10, scoring='accuracy')

print("Unregularized logistic regression (no penalty):")
print(f"  Training accuracy:      {train_acc:.4f}")
print(f"  CV accuracy (10-fold):  {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")
print()
print("Training accuracy is perfect (or near-perfect), but CV accuracy is much lower.")
print("This is the overfitting catastrophe: with p >> n, the model memorizes the training data.")

---
## Part 2: Ridge Regression

Ridge regression adds an L2 penalty: it shrinks all coefficients toward zero but keeps all genes in the model. No gene is ever exactly zeroed out.

In [ ]:
np.random.seed(2026)

# Ridge logistic regression with 10-fold CV, scoring by neg_log_loss for the curve.
# Lecture 4 uses CV deviance (= -2 * log-loss * n) for model selection; accuracy is too
# granular on n = 72 in 10 folds (~7 samples/fold) and masks the choice between Cs.
ridge = LogisticRegressionCV(
    penalty='l2',
    Cs=20,
    cv=10,
    max_iter=10000,
    random_state=2026,
    scoring='neg_log_loss'
)
ridge.fit(X, y)

# CV log-loss curve. scores_ is keyed by class label; for binary, take the positive class.
pos_class = ridge.classes_[1]
cs = ridge.Cs_
# neg_log_loss -> log-loss for plotting (smaller = better)
loss_per_fold = -ridge.scores_[pos_class]           # shape (n_folds, n_Cs)
mean_loss = loss_per_fold.mean(axis=0)
se_loss = loss_per_fold.std(axis=0) / np.sqrt(loss_per_fold.shape[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CV log-loss vs C with +/- 1 SE band
axes[0].semilogx(cs, mean_loss, 'o-', color='#4C72B0')
axes[0].fill_between(cs, mean_loss - se_loss, mean_loss + se_loss,
                     color='#4C72B0', alpha=0.2)
axes[0].axvline(x=ridge.C_[0], color='red', linestyle='--',
                label=f'Best C (min loss) = {ridge.C_[0]:.4f}')
axes[0].set_xlabel('C (inverse regularization strength)')
axes[0].set_ylabel('CV log-loss (lower is better)')
axes[0].set_title('Ridge: CV log-loss vs C  (mean ± 1 SE)')
axes[0].legend()

# Right: histogram of coefficient values at the best C
ridge_coefs = ridge.coef_.ravel()
axes[1].hist(ridge_coefs, bins=80, edgecolor='white', color='#4C72B0', alpha=0.8)
axes[1].set_xlabel('Coefficient value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Ridge: distribution of coefficients')

plt.tight_layout()
plt.show()

print(f"Best C (min CV log-loss): {ridge.C_[0]:.4f}  (smaller C = stronger penalty)")
print(f"Best CV log-loss:         {mean_loss.min():.4f}")
print(f"Coefficients exactly zero: {int(np.sum(ridge_coefs == 0))}  (ridge does not zero anything out)")
print(f"Coefficients |coef| > 0.01: {int(np.sum(np.abs(ridge_coefs) > 0.01))}")
print(f"Total coefficients:        {len(ridge_coefs)}")


### Exercises 2.1–2.3

**2.1** How many coefficients are exactly zero in the ridge fit? (Hint: ridge never zeros any coefficient out — that's the whole point of the $\ell_2$ penalty.)

**YOUR ANSWER:**

**2.2** What is the best $C$ from CV log-loss? Recall $C = 1/\lambda$, so smaller $C$ means stronger regularization.

**YOUR ANSWER:**

**2.3** Look at the CV log-loss curve. Is the minimum sharp, or is there a wide range of $C$ values with similar performance? If wide, the choice of $C$ within that range is essentially a tie — which has implications for the 1-SE rule in Part 3.

**YOUR ANSWER:**

---
## Part 3: LASSO

LASSO adds an L1 penalty, which gives sparse solutions — some coefficients become exactly zero. This performs variable selection: the genes that survive are the LASSO's chosen signature.

In [ ]:
np.random.seed(2026)

# LASSO logistic regression with 10-fold CV. saga solver supports L1.
# Scoring by neg_log_loss (CV deviance proxy) per Lecture 4.
lasso = LogisticRegressionCV(
    penalty='l1',
    solver='saga',
    Cs=20,
    cv=10,
    max_iter=10000,
    random_state=2026,
    scoring='neg_log_loss'
)
lasso.fit(X, y)

# Log-loss curve with +/- 1 SE band, plus 1-SE rule
pos_class = lasso.classes_[1]
cs_l1 = lasso.Cs_
loss_l1 = -lasso.scores_[pos_class]                  # (n_folds, n_Cs)
mean_l1 = loss_l1.mean(axis=0)
se_l1 = loss_l1.std(axis=0) / np.sqrt(loss_l1.shape[0])

# lambda_min in C-space: argmin of mean log-loss
i_min = int(np.argmin(mean_l1))
C_min = cs_l1[i_min]
loss_min = mean_l1[i_min]; se_at_min = se_l1[i_min]

# 1-SE rule: largest lambda (smallest C) within 1 SE of the minimum,
# corresponding to the most parsimonious model whose loss is within 1 SE of the best.
within_1se = mean_l1 <= loss_min + se_at_min
# Smaller C = stronger penalty = sparser model; pick the smallest C satisfying within_1se
candidates = np.where(within_1se)[0]
i_1se = int(candidates[np.argmin(cs_l1[candidates])]) if len(candidates) else i_min
C_1se = cs_l1[i_1se]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(cs_l1, mean_l1, 'o-', color='#DD8452')
ax.fill_between(cs_l1, mean_l1 - se_l1, mean_l1 + se_l1,
                color='#DD8452', alpha=0.2)
ax.axhline(loss_min + se_at_min, color='gray', linestyle=':', lw=1,
           label='1-SE band of the minimum')
ax.axvline(C_min, color='red', linestyle='--',
           label=f'$C_{{\min}}$ = {C_min:.4f}')
ax.axvline(C_1se, color='green', linestyle='--',
           label=f'$C_{{1\text{{SE}}}}$ = {C_1se:.4f}  (sparser, within 1 SE)')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('CV log-loss')
ax.set_title('LASSO: CV log-loss vs C  (mean ± 1 SE, with 1-SE rule)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Extract the sparser, more-defensible signature at C_1se
lr_1se = LogisticRegression(penalty='l1', solver='saga', C=C_1se,
                            max_iter=10000, random_state=2026).fit(X, y)
lasso_coefs = lr_1se.coef_.ravel()
nonzero_mask = lasso_coefs != 0
selected_genes = [gene_names[i] for i in range(len(gene_names)) if nonzero_mask[i]]
selected_coefs = lasso_coefs[nonzero_mask]

print(f"C_min  (best CV loss):              {C_min:.4f}    -> {int(np.sum(lasso.coef_.ravel() != 0))} genes")
print(f"C_1se  (sparser, within 1 SE):      {C_1se:.4f}    -> {len(selected_genes)} genes")
print(f"CV log-loss at C_min:  {loss_min:.4f}")
print(f"CV log-loss at C_1se:  {mean_l1[i_1se]:.4f}  (within 1 SE = {se_at_min:.4f})")
print(f"\nUsing the 1-SE signature ({len(selected_genes)} genes) for the rest of the lab:")
for g, c in sorted(zip(selected_genes, selected_coefs), key=lambda x: abs(x[1]), reverse=True):
    direction = "AML marker (y=1)" if c > 0 else "ALL marker (y=0)"
    print(f"  {g:>15s}: {c:+.4f}  ({direction})")


In [ ]:
# Bar chart of LASSO coefficients, colored by direction
order = np.argsort(np.abs(selected_coefs))[::-1]
ordered_genes = [selected_genes[i] for i in order]
ordered_coefs = selected_coefs[order]
colors = ['#DD8452' if c > 0 else '#4C72B0' for c in ordered_coefs]

fig, ax = plt.subplots(figsize=(10, max(4, len(ordered_genes) * 0.35)))
ax.barh(range(len(ordered_genes)), ordered_coefs, color=colors, edgecolor='white')
ax.set_yticks(range(len(ordered_genes)))
ax.set_yticklabels(ordered_genes, fontsize=10)
ax.set_xlabel('LASSO Coefficient')
ax.set_title('LASSO: Nonzero Coefficients (Orange = AML, Blue = ALL)')
ax.invert_yaxis()
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

### Exercises

**3.1** How many genes were selected by LASSO at $C_{\min}$? At $C_{1\text{SE}}$? Which would you report and why?

**YOUR ANSWER:**

**3.2** The 1-SE rule (Hastie, Tibshirani & Friedman, *ESL*) selects the most parsimonious model whose CV loss is within one standard error of the minimum. The figure above marks both $C_{\min}$ and $C_{1\text{SE}}$ explicitly. For *reporting* a gene signature (e.g., in a paper or as a diagnostic panel), which $C$ would you choose, and why?

**YOUR ANSWER:**

**3.3** *(Non-uniqueness caveat.)* When $p > n$ and the design matrix is not in general position — which is the case for gene-expression data with many correlated genes — the LASSO solution is generally *non-unique*: there can be a whole set of equivalent minimizers with different active sets ([Tibshirani 2013](https://doi.org/10.1214/13-EJS815)). This is one of two reasons the bootstrap stability analysis in Part 5 is essential (the other being sampling variability). Why doesn't ridge have this problem?

**YOUR ANSWER:**

---
## Part 4: Elastic Net

Elastic net combines L1 and L2 penalties — a compromise that handles correlated predictors better than LASSO alone. When genes are co-expressed, LASSO tends to pick one and drop the rest; elastic net keeps groups of correlated genes together.

In [ ]:
np.random.seed(2026)

# Elastic net with a real l1_ratio sweep (not a fixed point).
enet = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
    Cs=20,
    cv=10,
    max_iter=10000,
    random_state=2026,
    scoring='neg_log_loss'
)
enet.fit(X, y)

enet_coefs = enet.coef_.ravel()
enet_nonzero = enet_coefs != 0
enet_genes = set(gene_names[i] for i in range(len(gene_names)) if enet_nonzero[i])
lasso_genes_set = set(selected_genes)

overlap = lasso_genes_set & enet_genes
lasso_only = lasso_genes_set - enet_genes
enet_only = enet_genes - lasso_genes_set

# Mean log-loss across folds at the chosen (C, l1_ratio)
pos_class = enet.classes_[1]
mean_loss_enet = (-enet.scores_[pos_class]).mean(axis=0).ravel()
best_loss_enet = float(mean_loss_enet.min())

print(f"Elastic net best C:        {enet.C_[0]:.4f}")
print(f"Elastic net best l1_ratio: {enet.l1_ratio_[0]:.2f}  (sweep was 0.1, 0.3, 0.5, 0.7, 0.9)")
print(f"Elastic net CV log-loss:   {best_loss_enet:.4f}")
print(f"Elastic net genes selected (at best C and l1_ratio): {len(enet_genes)}")
print()
print(f"Overlap with LASSO 1-SE signature ({len(lasso_genes_set)} genes):")
print(f"  Both LASSO and elastic net: {len(overlap)}")
print(f"  LASSO only:                 {len(lasso_only)}")
print(f"  Elastic net only:           {len(enet_only)}")

if lasso_only:
    print(f"\nLASSO-only genes: {sorted(lasso_only)}")
if enet_only:
    print(f"Elastic-net-only genes: {sorted(enet_only)}")


In [ ]:
# Compare coefficient magnitudes for shared genes
if overlap:
    shared_genes_list = sorted(overlap)
    gene_to_idx = {g: i for i, g in enumerate(gene_names)}
    lasso_shared = [lasso_coefs[gene_to_idx[g]] for g in shared_genes_list]
    enet_shared = [enet_coefs[gene_to_idx[g]] for g in shared_genes_list]

    fig, ax = plt.subplots(figsize=(8, 5))
    x_pos = np.arange(len(shared_genes_list))
    width = 0.35
    ax.bar(x_pos - width/2, lasso_shared, width, label='LASSO', color='#DD8452', edgecolor='white')
    ax.bar(x_pos + width/2, enet_shared, width, label='Elastic Net', color='#55A868', edgecolor='white')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(shared_genes_list, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Coefficient')
    ax.set_title('Shared Genes: LASSO vs Elastic Net Coefficients')
    ax.legend()
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()

### Exercise

**4.1** Does elastic net select more or fewer genes than LASSO? Explain why, based on the properties of the L1 + L2 penalty.

**YOUR ANSWER:**

---
## Part 5: Stability Analysis

LASSO selection is unstable: small changes in the data can flip genes in or out of the model. Bootstrap resampling reveals which genes are *consistently* selected. These are the ones you should trust.

In [ ]:
np.random.seed(2026)

# Bootstrap stability: 100 resamples is enough for a stable selection-frequency estimate.
# CRITICAL: standardize INSIDE each bootstrap iteration (on the resample only). Standardizing
# once on the full data and reusing X across resamples leaks information across folds — exactly
# the textbook error Lecture 4 warns about. We also use the same solver (saga) as the LASSO CV
# in Part 3 so that the active sets are computed on the same optimization problem.
n_boot = 100
n_samples = X_raw.shape[0]
n_genes = X_raw.shape[1]
selection_counts = np.zeros(n_genes)

# Reuse the 1-SE C from Part 3 — the sparser, more-defensible signature.
C_boot = C_1se

print(f"Running {n_boot} bootstrap LASSO fits at C = {C_boot:.4f} (1-SE rule from Part 3)...")
for b in range(n_boot):
    # Bootstrap sample of indices, with replacement
    idx = np.random.choice(n_samples, size=n_samples, replace=True)
    X_boot_raw = X_raw[idx]
    y_boot = y[idx]

    # Refit StandardScaler on this resample (do NOT use the global X)
    X_boot = StandardScaler().fit_transform(X_boot_raw)

    lr_boot = LogisticRegression(
        penalty='l1',
        solver='saga',
        C=C_boot,
        max_iter=10000,
        random_state=2026,
        tol=1e-3,            # slightly looser tolerance keeps the loop fast at n_boot=100
    )
    lr_boot.fit(X_boot, y_boot)

    selection_counts += (lr_boot.coef_.ravel() != 0).astype(int)

    if (b + 1) % 20 == 0:
        print(f"  Completed {b + 1}/{n_boot}")

selection_freq = selection_counts / n_boot

top_idx = np.argsort(selection_freq)[::-1][:20]
top_genes = [gene_names[i] for i in top_idx]
top_freqs = selection_freq[top_idx]

fig, ax = plt.subplots(figsize=(10, 7))
colors_boot = ['#DD8452' if f >= 0.5 else '#999999' for f in top_freqs]
ax.barh(range(len(top_genes)), top_freqs, color=colors_boot, edgecolor='white')
ax.set_yticks(range(len(top_genes)))
ax.set_yticklabels(top_genes, fontsize=10)
ax.set_xlabel('Selection Frequency')
ax.set_title(f'Bootstrap stability (n_boot = {n_boot}; standardization inside the loop)')
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

# Summary counts
n_50 = int(np.sum(selection_freq > 0.5))
n_80 = int(np.sum(selection_freq > 0.8))
n_90 = int(np.sum(selection_freq > 0.9))
print(f"Genes selected in >50% of bootstraps: {n_50}")
print(f"Genes selected in >80% of bootstraps: {n_80}")
print(f"Genes selected in >90% of bootstraps: {n_90}")


### Critique checklist

| Question | Pass / Fail / Partial | Notes |
|---|---|---|
| Did the bootstrap loop standardize *inside* each iteration (no leakage from the full data)? | | |
| Did the bootstrap use the same solver as the LASSO CV (saga) for comparable optima? | | |
| Did the bootstrap use sampling with replacement (`replace=True`)? | | |
| How many genes were selected in >50% of the 100 bootstrap fits? | | |
| Are the most stable genes a subset of the largest-magnitude LASSO coefficients, or a different set? | | |

### Exercise

**5.1** A colleague says "LASSO selected 15 genes, so these *cause* the disease." Two things are wrong with this. (i) LASSO does not establish causation. (ii) Even the gene *list* is not identifiable: in $p > n$, the LASSO active set is generally non-unique under correlated columns (Exercise 3.3), and the selected list shifts under resampling. Use the bootstrap-stability frequencies you just computed to argue that only the high-frequency subset (e.g., >50%) is a defensible "signature."

**YOUR ANSWER:**


### Exercise 5.2 — Multi-turn: interpret the bootstrap-stability plot *(required multi-turn)*

This is the multi-turn AI exercise for Lab 4. **Three turns of model output in the same chat session** so the AI can build on (and revise) its earlier answer.

**Turn 1 — initial prompt.** Ask the AI:

> "Reply in three short paragraphs only (no .docx file). I ran a LASSO logistic-regression analysis on a 72-sample microarray dataset to distinguish ALL from AML leukemia, using the 1-SE rule for $\lambda$ selection and standardizing inside each bootstrap iteration. I then bootstrapped the analysis (100 resamples with replacement) and recorded how often each gene was selected. The plot shows the top-20 genes by selection frequency: some appear in 90–100% of bootstraps, others appear in 30–60%. Which genes should I report in a paper, and why? What does the stability spectrum tell me about the underlying signal?"

**AI's response (Turn 1, paste here):**


**Your critique (write your own — do not copy the bullets below verbatim).** Common failure modes to look for:

- The AI says "report the 50%+ genes" without explaining *why* — i.e., without naming LASSO selection instability under correlated predictors.
- The AI does not distinguish between *sampling-variability* instability and *non-uniqueness* of the LASSO optimum.
- The AI does not interpret the 30–60% genes correctly — those are plausibly in the signal but the LASSO is forced to pick one of each correlated cluster arbitrarily.
- The AI does not propose a remedy (elastic net with the grouping property; ridge for confirmation; biological validation by qPCR).

**YOUR CRITIQUE OF THE TURN-1 RESPONSE:**


**Turn 2 — your follow-up prompt.** Based on the specific weaknesses you flagged, write a prompt in your own words that asks the AI to revise its answer. *In the same chat session*, send it. Don't copy a canned phrase — write what *you* want explained.

**YOUR TURN-2 PROMPT:**

**AI's response (Turn 2):**


**Turn 3 — actionable.** In the same chat session, ask the AI to predict *empirically* what would change if you reran the bootstrap with elastic net at $\alpha = 0.5$ instead of pure LASSO. Specifically: would the total number of selected genes go up or down, and would individual stability frequencies go up or down? The grouping property predicts that correlated co-regulated genes are now selected *together*, so the total selected count goes up *and* the individual frequencies of correlated genes go up.

**YOUR TURN-3 PROMPT:**

**AI's response (Turn 3):**


**Final assessment.** Did the Turn-3 response correctly predict the direction of change under elastic net (more selected genes, higher individual stabilities for correlated clusters)? If still hand-wavy, what is the right one-paragraph explanation in your own words?

---
## Part 6: Model Comparison

Compare all three methods side by side. Same data, same cross-validation, different penalties.

In [ ]:
np.random.seed(2026)

# Compare using the internal CV log-loss from each fitted model.
# IMPORTANT: these are *training-CV* losses — the C (and l1_ratio for enet) were tuned on
# the same folds being reported, so the numbers are optimistic estimates of test
# performance. A proper unbiased estimate requires nested CV (outer CV wrapping the
# inner CV used for tuning). We accept the bias for in-class comparison and flag it
# explicitly in the printed output.

pos_class_ridge = ridge.classes_[1]
pos_class_lasso = lasso.classes_[1]
pos_class_enet = enet.classes_[1]

ridge_min_loss = float((-ridge.scores_[pos_class_ridge]).mean(axis=0).ravel().min())
lasso_min_loss = float((-lasso.scores_[pos_class_lasso]).mean(axis=0).ravel().min())
enet_min_loss = float((-enet.scores_[pos_class_enet]).mean(axis=0).ravel().min())

# Active set sizes
n_nonzero_ridge = int(np.sum(ridge.coef_.ravel() != 0))
n_nonzero_lasso_min = int(np.sum(lasso.coef_.ravel() != 0))    # at C_min
n_nonzero_lasso_1se = int(np.sum(selected_coefs != 0))         # at C_1se (Part 3)
n_nonzero_enet = int(np.sum(enet.coef_.ravel() != 0))

comparison = pd.DataFrame({
    'Method':        ['Ridge (L2)', 'LASSO (L1, C_min)', 'LASSO (L1, C_1se)', 'Elastic Net'],
    'CV log-loss':   [ridge_min_loss, lasso_min_loss, np.nan, enet_min_loss],
    'Nonzero genes': [n_nonzero_ridge, n_nonzero_lasso_min, n_nonzero_lasso_1se, n_nonzero_enet],
})

print(comparison.to_string(index=False))
print()
print("Caveat: CV log-loss values come from LogisticRegressionCV's internal cross-validation,")
print("which selected the hyperparameter on the same folds. The reported losses are therefore")
print("optimistic estimates of out-of-sample performance. For a clinical signature you would")
print("estimate test performance via nested CV (outer CV for evaluation; inner CV for tuning)")
print("or against an external validation cohort — as MammaPrint did in MINDACT (Lecture 4).")


### Exercise

**6.1** If all three methods have similar accuracy but very different gene counts, what does this tell you about the data?

**YOUR ANSWER:**

---
## Part 7: Heatmap of Stable Genes

Visualize the stably selected genes across all 72 samples. If LASSO found a real signature, these genes should visually separate ALL from AML samples.

In [ ]:
# Get genes selected in >50% of bootstraps
stable_idx = np.where(selection_freq > 0.5)[0]
stable_genes = [gene_names[i] for i in stable_idx]

if len(stable_genes) == 0:
    print("No genes above 50% threshold. Lowering to 30%.")
    stable_idx = np.where(selection_freq > 0.3)[0]
    stable_genes = [gene_names[i] for i in stable_idx]

print(f"Stable genes for heatmap: {len(stable_genes)}")

# Extract expression values (samples x selected genes), re-standardized
sample_labels = [f"S{i}" for i in range(X.shape[0])]
heatmap_data = pd.DataFrame(
    X[:, stable_idx],
    columns=stable_genes,
    index=sample_labels
).T

# Color annotation for ALL/AML subtype
subtype_labels = labels.values
subtype_map = {'ALL': '#4C72B0', 'AML': '#DD8452'}
col_colors = [subtype_map[s] for s in subtype_labels]

g = sns.clustermap(
    heatmap_data,
    cmap='RdBu_r',
    center=0,
    col_colors=col_colors,
    figsize=(14, max(6, len(stable_genes) * 0.4)),
    dendrogram_ratio=(0.1, 0.15),
    cbar_pos=(0.02, 0.8, 0.03, 0.15),
    yticklabels=True,
    xticklabels=False
)
g.ax_heatmap.set_xlabel('Samples (Blue = ALL, Orange = AML)')
g.ax_heatmap.set_ylabel('Genes')
g.fig.suptitle('Stably Selected Genes Across Leukemia Samples', y=1.02, fontsize=14)
plt.show()

---
## Part 8: AI Extension

Ask an AI assistant (the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, or ChatGPT) to compare your gene signature to known leukemia biology.

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file — just runnable code I can paste into a Colab cell). For each gene in the list below, look up its function and relevance to ALL/AML biology and assemble a `pandas.DataFrame` with columns `gene`, `function`, `pathway`, `relevance_to_leukemia`, and `citation_DOI`. If you cannot find a citation, use the literal string `'unknown'` in the citation column. Do not fabricate DOIs. Genes: [paste stable gene list from the cell below]."

**Why the format directive matters.** Without it, some AI platforms (LibreChat in particular) wrap long answers in a Word document — which cannot be pasted into Colab. The directive returns a paste-ready code block.

**Why the structured DataFrame matters.** A structured output is easier to verify than freeform prose. The next cell asks you to spot-check at least three rows against [GeneCards](https://www.genecards.org/) or [NCBI Gene](https://www.ncbi.nlm.nih.gov/gene/) — fabricated citations and incorrect function descriptions are a common AI failure mode in this domain (mechanisms confused, gene names invented, DOIs hallucinated). Asking for `'unknown'` in place of fabricated citations gives the AI an explicit escape hatch from making things up.

In [ ]:
# Print stable gene list for easy copy-paste to AI
print("Stable gene list (>50% bootstrap frequency):")
print(", ".join(stable_genes))

In [ ]:
# YOUR CODE HERE (AI-generated or your own)


### Verification Table

Verify at least 3 genes using [GeneCards](https://www.genecards.org/) or [NCBI Gene](https://www.ncbi.nlm.nih.gov/gene/).

| Gene | AI's claim | What GeneCards/NCBI says | Accurate? |
|------|-----------|------------------------|----------|
| | | | |
| | | | |
| | | | |

**Did the AI hallucinate any gene functions?**

**YOUR ANSWER:**

---
## Part 9: Connecting Day 1 and Day 2

**9.1** How could you combine FDR analysis (Day 1) with LASSO (today) in a single pipeline?

**YOUR ANSWER:**

**9.2** Would you expect the BH-FDR gene list and LASSO gene list to overlap completely? Why or why not?

(Hint: FDR finds differential expression; LASSO finds prediction. Redundant genes may be FDR-significant but dropped by LASSO because they add no predictive value beyond what correlated genes already provide.)

**YOUR ANSWER:**

---

## Wrap-Up Questions

1. When would you choose ridge over LASSO in a genomics application? When would you choose elastic net?

   **YOUR ANSWER:**

2. The Cells-from-correlated-clusters problem is the *central* motivation of Lecture 4 ("The Coefficients That Stayed Zero"). What is the relationship between LASSO non-uniqueness in $p > n$ (Exercise 3.3) and bootstrap-selection instability (Part 5)?

   **YOUR ANSWER:**

3. Looking at the validation pipelines table from the lecture (MammaPrint MINDACT vs Duke metagene), which step of *this lab's* pipeline would you change to make the gene signature defensible for clinical use?

   **YOUR ANSWER:**

4. What did you learn verifying the AI's gene descriptions against GeneCards in Part 8?

   **YOUR ANSWER:**